In [1]:
import numpy as np

PATH_TRAIN = 'data/processed/new_handmarks.npy'
PATH_TEST = 'data/processed/new_targets.npy'

X = np.load(PATH_TRAIN, allow_pickle=True)
y = np.load(PATH_TEST, allow_pickle=True)

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, 
                                                    stratify = y, random_state=42)


In [6]:
from sklearn.preprocessing import MinMaxScaler

mms = MinMaxScaler(feature_range=(-1, 1))

X_1d = X_train.reshape(-1)
X_2d = np.array([item for lst in X_1d for arr in lst for item in arr]).reshape(-1, 1)

mms.fit(X_2d)

MinMaxScaler(feature_range=(-1, 1))

In [26]:
#HERE I JUST PREPROCESS ALL FRAMES AND VIDEOS -
#THE IS A PADDING FOR VIDEOS AND FRAME; VIDEOS CONSIST OF 25 FRAMES SPACED ACCROSS VIDEO
#I ALSO DO SCALING (-1, 1) FOR TRAIN AND TEST DATA
import numpy as np


N_FRAMES = 25

def into_normal_arrays_with_padding(X_id_):
    X_train_processed = []
    for video in X_id_:
        k = max(1, len(video) // N_FRAMES)
        sampled_frames = []
        for i in range(0, len(video), k):
            if len(sampled_frames) >= N_FRAMES:
                break
            frame = mms.transform(np.array(video[i]).reshape(-1, 1))
            frame = frame.reshape(-1)
            if len(frame) < 128:
                frame = np.pad(frame, (0, 128 - len(frame)), mode='constant')
            elif len(frame) > 128:
                frame = frame[:128]
            sampled_frames.append(np.array(frame, dtype=np.float32))
        while len(sampled_frames) < N_FRAMES:
            sampled_frames.append([0.0] * 128)
        X_train_processed.append(np.array(sampled_frames))
    return np.array(X_train_processed)
        

X_train_processed = into_normal_arrays_with_padding(X_train)
X_test_processed = into_normal_arrays_with_padding(X_test)
print(f"Final shape: {X_train_processed.shape}")

Final shape: (17000, 25, 128)


In [10]:
print(min(X_train[1][0]), max(X_train[1][0]))

[0.169, 0.843, 0.0, 0.243, 0.842, -0.016, 0.299, 0.851, -0.04, 0.333, 0.871, 
 -0.061, 0.352, 0.894, -0.083, 0.253, 0.837, -0.065, 0.265, 0.863, -0.101, 0.296, 
 0.886, -0.127, 0.32, 0.903, -0.144, 0.209, 0.85, -0.075, 0.227, 0.886, -0.11, 0.263,
 0.911, -0.122, 0.293, 0.926, -0.133, 0.172, 0.864, -0.085, 0.192, 0.902, -0.12, 0.227, 
 0.924, -0.123, 0.254, 0.937, -0.123, 0.139, 0.881, -0.096, 0.163, 0.915, -0.12, 0.196, 
 0.933, -0.121, 0.223, 0.942, -0.121]  #21 * 3 -> frame X_train[x][y]
                                        #frame * n (кадров) -> video X_train[x]
                                        #video * k -> X_train

-0.046 0.891


[0.169,
 0.843,
 0.0,
 0.243,
 0.842,
 -0.016,
 0.299,
 0.851,
 -0.04,
 0.333,
 0.871,
 -0.061,
 0.352,
 0.894,
 -0.083,
 0.253,
 0.837,
 -0.065,
 0.265,
 0.863,
 -0.101,
 0.296,
 0.886,
 -0.127,
 0.32,
 0.903,
 -0.144,
 0.209,
 0.85,
 -0.075,
 0.227,
 0.886,
 -0.11,
 0.263,
 0.911,
 -0.122,
 0.293,
 0.926,
 -0.133,
 0.172,
 0.864,
 -0.085,
 0.192,
 0.902,
 -0.12,
 0.227,
 0.924,
 -0.123,
 0.254,
 0.937,
 -0.123,
 0.139,
 0.881,
 -0.096,
 0.163,
 0.915,
 -0.12,
 0.196,
 0.933,
 -0.121,
 0.223,
 0.942,
 -0.121]

In [ ]:
#WORK ON DECOMPOSITION LATER
from sklearn.random_projection import GaussianRandomProjection
from sklearn.pipeline import Pipeline

grp = GaussianRandomProjection(
    n_components = 0.9,
    copy = True,
    eps = 0.1,
    random_state = 42
)

In [27]:
print(X_train_processed.shape, X_test_processed.shape)
y_train_int.shape, y_test.shape

(17000, 25, 128) (3000, 25, 128)


((17000,), (3000,))

In [30]:
from tensorflow.keras.layers import BatchNormalization, LSTM, Masking, Dropout, Dense
from tensorflow.keras.models import Sequential

model = Sequential([
    Masking(mask_value=0.0, input_shape=(None, 128)),
    LSTM(128, return_sequences=True, dropout=0.2),
    BatchNormalization(),  # Ускоряет обучение
    LSTM(64, dropout=0.2),
    BatchNormalization(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1000, activation='softmax')
])

In [ ]:
from sklearn.preprocessing import LabelEncoder
# ПРОСТО ПРЕОБРАЗУЙ СТРОКИ В ЦИФРЫ
label_encoder = LabelEncoder()
y_train_int = label_encoder.fit_transform(y_train)
y_test_int = label_encoder.transform(y_test)

In [32]:

from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.01),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train_processed,
    y_train_int,
    epochs=50,
    batch_size=32,
    validation_data=(X_test_processed, y_test_int),
    verbose=1
)

Epoch 1/50
532/532 [==============================] - 17s 28ms/step - loss: 6.8409 - accuracy: 0.0028 - val_loss: 6.8835 - val_accuracy: 0.0040
Epoch 2/50
532/532 [==============================] - 14s 26ms/step - loss: 6.2993 - accuracy: 0.0063 - val_loss: 6.4465 - val_accuracy: 0.0037
Epoch 3/50
532/532 [==============================] - 14s 26ms/step - loss: 6.0428 - accuracy: 0.0087 - val_loss: 5.8620 - val_accuracy: 0.0110
Epoch 4/50
532/532 [==============================] - 14s 26ms/step - loss: 5.8492 - accuracy: 0.0136 - val_loss: 6.5098 - val_accuracy: 0.0077
Epoch 5/50
532/532 [==============================] - 14s 26ms/step - loss: 5.6789 - accuracy: 0.0184 - val_loss: 5.6522 - val_accuracy: 0.0190
Epoch 6/50
532/532 [==============================] - 14s 26ms/step - loss: 5.5330 - accuracy: 0.0225 - val_loss: 5.4976 - val_accuracy: 0.0273
Epoch 7/50
532/532 [==============================] - 14s 26ms/step - loss: 5.4043 - accuracy: 0.0282 - val_loss: 5.7420 - val_accuracy:

KeyboardInterrupt: 